# veckit — Virtual Embryo Challenge local scorer (tutorial)

`veckit` scores a submitted `.h5ad` against the T1/T2/T3 metric panels **entirely offline**, against
reference files *you* supply — no access to the official (held-out) validation/test data, ever. It's a
real, standalone PyPI package: [pypi.org/project/veckit](https://pypi.org/project/veckit/).

**Three files, three roles**
| flag | role |
|---|---|
| `--input` | the prediction you're scoring |
| `--target` | the pseudo target — what `--input` should have predicted |
| `--reference` (T1/T2) / `--wt` (T3) | the reference expression `de_score`/`de_direction`/`severity_slope` are measured *from* — these are PRIMARY metrics ("did you predict the right **change**", not just "does this look plausible"), and a change can't be computed from a single snapshot |

`--reference`/`--wt` is optional and defaults to `--input` itself if omitted — correct **only** when
you're deliberately testing a no-change baseline (`copy_last`/`wt_identity`), where it exactly reproduces
the official floor numbers. For a real model's prediction, always pass a genuine reference file, or
`de_score`/`de_direction` will misleadingly read as "no predicted change" (exactly 0) regardless of what
your model actually did.

**This is not a preview of your real competition score.** Whatever you pass as `--target` is, by
definition, data you already had, so it's typically an easier question than the real held-out target.

## Install

In [ ]:
!pip install -q veckit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 98.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.7/363.7 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 116.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
numba 0.60.0 requires 

## Get the data

**For a real local score**, register and download the released T1/T2/T3 stages from the official
challenge site — [virtualembryo.ai/challenge/data](https://virtualembryo.ai/challenge/data) (see also
[aristoteleo/virtualembryo](https://github.com/aristoteleo/virtualembryo)) — then point `--input`/
`--target`/`--reference`/`--wt` at your own downloaded files.

**To just try the tool first**, this repo ships a few tiny (150-cell) samples — enough to exercise the
CLI, not enough to mean anything scientifically:

In [ ]:
!mkdir -p sample_data
!wget -q -O sample_data/T1_8.5.h5ad  https://raw.githubusercontent.com/aristoteleo/veckit/main/data/sample_8.5.h5ad
!wget -q -O sample_data/T1_9.5.h5ad  https://raw.githubusercontent.com/aristoteleo/veckit/main/data/sample_9.5.h5ad
!wget -q -O sample_data/T2_9.25.h5ad https://raw.githubusercontent.com/aristoteleo/veckit/main/data/sample_heart_9.25.h5ad
!wget -q -O sample_data/T2_9.5.h5ad  https://raw.githubusercontent.com/aristoteleo/veckit/main/data/sample_heart_9.5.h5ad
!wget -q -O sample_data/T3_wt.h5ad   https://raw.githubusercontent.com/aristoteleo/veckit/main/data/sample_wt.h5ad
!wget -q -O sample_data/T3_ko.h5ad   https://raw.githubusercontent.com/aristoteleo/veckit/main/data/sample_mab21l2_ko.h5ad
!ls -la sample_data/

total 71852
drwxr-xr-x 1 root root     4096 Aug 10 19:04 .
drwxr-xr-x 1 root root     4096 Jun  4 13:32 ..
-rwxr-xr-x 1 root root     1697 Jan  1  2000 anscombe.json
-rw-r--r-- 1 root root   301141 Jun  4 13:32 california_housing_test.csv
-rw-r--r-- 1 root root  1706430 Jun  4 13:32 california_housing_train.csv
-rw-r--r-- 1 root root 18289443 Jun  4 13:32 mnist_test.csv
-rw-r--r-- 1 root root 36523880 Jun  4 13:32 mnist_train_small.csv
-rwxr-xr-x 1 root root      962 Jan  1  2000 README.md
-rw-r--r-- 1 root root  6650592 Aug 10 19:03 T1_8.5.h5ad
-rw-r--r-- 1 root root  6805136 Aug 10 19:03 T1_9.5.h5ad
-rw-r--r-- 1 root root   560816 Aug 10 19:03 T2_9.25.h5ad
-rw-r--r-- 1 root root   565224 Aug 10 19:03 T2_9.5.h5ad
-rw-r--r-- 1 root root  1574024 Aug 10 19:04 T3_ko.h5ad
-rw-r--r-- 1 root root   562744 Aug 10 19:04 T3_wt.h5ad


## Task 1 — temporal gene-expression prediction

`--reference` is the genuine preceding stage (E8.5), so `de_score`/`de_direction` are computed
meaningfully, not defaulted. `--input` here is E9.5 itself, standing in for a hypothetically *perfect*
prediction — replace it with your own model's output file.

In [ ]:
!veckit --task T1 \
  --input sample_data/T1_9.5.h5ad \
  --target sample_data/T1_9.5.h5ad \
  --reference sample_data/T1_8.5.h5ad

{
  "meta": {
    "task": "T1",
    "target_source": "sample_data/T1_9.5.h5ad",
    "reference_defaulted_to_input": false,
    "truth_cells": 150,
    "prediction_cells": 150,
    "genes": 32285
  },
  "metrics": {
    "de_score": 0.5152,
    "de_direction": 1.0,
    "energy_distance": -0.8259,
    "mmd_u": -0.00813,
    "variogram": 0.0,
    "pb_rel_err": 0.0,
    "library_size_ratio": 1.0,
    "variance_ratio": 1.0,
    "composition_JSD": 0.0,
    "pseudobulk_pearson": 1.0,
    "_de_raw": 0.5152,
    "_de_chance": 0.0,
    "_de_chance_unif": 0.001,
    "_n_up": 18,
    "_n_dn": 15
  }
}


**Quick check, no explicit `--reference`.** Useful for a fast copy_last-style sanity check on the pipeline itself; `--input` doubles as the reference, so `de_score`/`de_direction` correctly (not misleadingly) read as 0 here:

In [ ]:
!veckit --task T1 --input sample_data/T1_8.5.h5ad --target sample_data/T1_9.5.h5ad

[veckit] --reference not given -- defaulting to --input itself. de_score/de_direction will correctly read as 'no predicted change', which is only meaningful if you're testing a no-change baseline (copy_last/wt_identity). Pass --reference explicitly for a real model.
{
  "meta": {
    "task": "T1",
    "target_source": "sample_data/T1_9.5.h5ad",
    "reference_defaulted_to_input": true,
    "truth_cells": 150,
    "prediction_cells": 150,
    "genes": 32285
  },
  "metrics": {
    "de_score": 0.0,
    "de_direction": 0.0,
    "energy_distance": 0.36497,
    "mmd_u": 0.07716,
    "variogram": 0.00187,
    "pb_rel_err": 0.1211,
    "library_size_ratio": 1.0,
    "variance_ratio": 0.974,
    "composition_JSD": 0.0266,
    "pseudobulk_pearson": 0.9915,
    "_de_raw": 0.0,
    "_de_chance": 0.0,
    "_de_chance_unif": 0.001,
    "_n_up": 18,
    "_n_dn": 15
  }
}


## Task 2 — spatial-temporal multiscale prediction

Same three roles, plus `obsm['spatial_3D']` on every file (already present in the sample data) and
`--setting` (`heart`/`embryo`) to label which scope you're scoring.

In [ ]:
!veckit --task T2 --setting heart \
  --input sample_data/T2_9.5.h5ad \
  --target sample_data/T2_9.5.h5ad \
  --reference sample_data/T2_9.25.h5ad

{
  "meta": {
    "task": "T2",
    "setting": "heart",
    "target_source": "sample_data/T2_9.5.h5ad",
    "reference_defaulted_to_input": false,
    "truth_cells": 150,
    "prediction_cells": 150,
    "genes": 500
  },
  "metrics": {
    "de_score": 0.5102,
    "de_direction": 1.0,
    "energy_distance": -0.51577,
    "mmd_u": -0.00807,
    "variogram": 0.0,
    "d2_shape": 0.00126,
    "sliced_wasserstein": 0.0,
    "occupancy_dice": 1.0,
    "scale_log_ratio": 0.0,
    "count_log_ratio": 0.0,
    "neighborhood_mmd": -0.00793,
    "pb_rel_err": 0.0,
    "library_size_ratio": 1.0,
    "variance_ratio": 1.0,
    "composition_JSD": 0.0,
    "pseudobulk_pearson": 1.0,
    "morans_I_agreement": 1.0,
    "_de_raw": 0.7209,
    "_de_chance": 0.4302,
    "_n_up": 86,
    "_n_dn": 0,
    "_sw_flip_spread": 0.08091,
    "_dice_voxel_over_nn": 1.9
  }
}


## Task 3 — mutant perturbation prediction

`--wt` plays `--reference`'s role here: the matched wild type the knockout effect (`de_score`/
`de_direction`/`severity_slope`) is measured against.

In [ ]:
!veckit --task T3 \
  --input sample_data/T3_ko.h5ad \
  --target sample_data/T3_ko.h5ad \
  --wt sample_data/T3_wt.h5ad

{
  "meta": {
    "task": "T3",
    "target_source": "sample_data/T3_ko.h5ad",
    "wt_defaulted_to_input": false,
    "truth_cells": 150,
    "prediction_cells": 150,
    "genes": 500
  },
  "metrics": {
    "de_score": 0.641,
    "de_direction": 1.0,
    "severity_slope": 0.0,
    "energy_distance": -0.53215,
    "mmd_u": -0.00804,
    "variogram": 0.0,
    "pb_rel_err": 0.0,
    "library_size_ratio": 1.0,
    "variance_ratio": 1.0,
    "composition_JSD": 0.0,
    "pseudobulk_pearson": 1.0,
    "_de_raw": 0.6818,
    "_de_chance": 0.1136,
    "_n_up": 35,
    "_n_dn": 9,
    "_slope_r2": 1.0,
    "d2_shape": 0.00273,
    "sliced_wasserstein": 0.0,
    "occupancy_dice": 1.0,
    "scale_log_ratio": 0.0,
    "count_log_ratio": 0.0,
    "neighborhood_mmd": -0.00799,
    "_sw_flip_spread": 0.05966,
    "_dice_voxel_over_nn": 2.0
  }
}


## Python API

Same thing, without shelling out — useful inside a training/eval loop. `from veckit import score` mirrors
every CLI flag 1:1 as a keyword argument.

In [ ]:
from veckit import score

result = score(
    task="T1",
    input="sample_data/T1_8.5.h5ad",
    target="sample_data/T1_9.5.h5ad",
    reference="sample_data/T1_8.5.h5ad",
)
result["metrics"]

{'de_score': 0.0,
 'de_direction': 0.0,
 'energy_distance': 0.36497,
 'mmd_u': 0.07716,
 'variogram': 0.00187,
 'pb_rel_err': 0.1211,
 'library_size_ratio': 1.0,
 'variance_ratio': 0.974,
 'composition_JSD': 0.0266,
 'pseudobulk_pearson': 0.9915,
 '_de_raw': 0.0,
 '_de_chance': 0.0,
 '_de_chance_unif': 0.001,
 '_n_up': 18,
 '_n_dn': 15}

## Next steps

- Swap `--input` for your own model's prediction (or `input=` in Python).
- Download the real released stages from
  [virtualembryo.ai/challenge/data](https://virtualembryo.ai/challenge/data) and swap `--target`/
  `--reference`/`--wt` for them.
- `veckit --help` / `help(score)` for every flag.
- Full task definitions and metric rationale: [virtualembryo.ai](https://virtualembryo.ai).